# V-MD3 Stepped SAR Acquisition

Records stepped SAR scans with the V-MD3 61 GHz FMCW radar mounted on a Galil-driven linear stage. The stage starts exactly on one edge, steps across the aperture in equal steps, and stops at each position to record a set number of frames. Each position is saved to its own file, and the encoder position is logged at every stop.

**Hardware**

- V-MD3 radar on Ethernet (static IP `192.168.100.201`)
- Griffin Motion linear stage on a Galil DMC-30012 controller over USB (COM port)
- Stage scale is 125 nm per count with about 101 mm of travel between limit switches

**Code layout**

The device code lives in `lib/` and normally needs no edits. Each notebook cell holds the settings it uses, right above the call.

- `lib/radar_link.py`: `Radar` class and the network setup
- `lib/galil_stage.py`: `Stage` class (moves, homing, calibration)
- `lib/sar_sweep.py`: scan planner, stepped recorder, logs, and `.bin` export

**One-time install**

1. Install gclib from Galil's website using the `.exe` installer.
2. In PowerShell, install the Python wrapper:
   `py -m pip install https://www.galil.com/sw/pub/python/gclib-1.0.1-py3-none-any.whl`
   
   Or if using a venv, run this in a notebook cell:
   `%pip install https://www.galil.com/sw/pub/python/gclib-1.0.1-py3-none-any.whl`

Use wrapper version 1.0.1. The newer 2.0 wrapper has a different API that `lib/galil_stage.py` doesn't use.

**Each session**

1. Close GDK so the COM port is free.
2. Run the imports cell below.
3. Run **1a** (only after plugging in or power cycling the radar), then **1b**.
4. Run **2** to connect the stage and home it to the start edge.
5. Run **3** to plan the scan.

**Each scan:** set `SCAN_NAME` in **Section 4** and run it. Re-run Section 3 first if you change the plan.

**When finished:** run **Section 5**. Section 6 (export) and the Appendix (calibration) are optional.

In [ ]:
%load_ext autoreload
%autoreload 2

from lib.radar_link import Radar, run_network_setup
from lib.galil_stage import Stage
from lib.sar_sweep import plan_steps, record_stepped, export_long_csv

## 1. Radar

### 1a. Network setup

Only needed after plugging in or power cycling the radar. Opens a UAC prompt, then prints the setup script's checks.

In [ ]:
RADAR_SETUP_SCRIPT = "C:/My Computer/vmd3/setup_radar.ps1"

run_network_setup(RADAR_SETUP_SCRIPT)

### 1b. Connect and measure frame period

Connects, configures the radar, and times 30 frames. The frame period is used for the scan time estimate. Re-run after changing `RADAR_MODE` or `RADAR_STREAMS`.

If it hangs after "Connected to UDP", power cycle the radar, wait about 10 s, then run 1a and 1b again.

In [ ]:
RADAR_IP = "192.168.100.201"
RADAR_MODE = 0   # RSET 0 = 2D, 6 m max range, 4.69 cm range bins
RADAR_STREAMS = ["radc"]   # any of "radc", "rfft", "done"

# Close the previous connection if this cell is re-run
if "radar" in globals():
    radar.close()

radar = Radar(RADAR_IP)
radar.connect(RADAR_MODE, RADAR_STREAMS)
radar.measure_frame_period()

## 2. Stage

Connects to the controller and homes the stage to the start edge. With `HOME_BACKOFF_COUNTS = 0`, the stage stops exactly where the start-side limit switch trips, and that point becomes position 0 of every scan. A back-off above 0 moves position 0 that far off the edge instead. Close GDK first. Re-run if the stage is bumped or power cycled.

`START_DIRECTION = +1` starts at the right edge and scans left. Use `-1` to start at the left edge and scan right.

You may see a "timed out, reconnecting" message when the stage reaches the switch. That's a harmless comms hiccup the stage recovers from on its own.

In [ ]:
STAGE_PORT = "COM4 --baud 115200 -d"
COUNTS_PER_MM = 8_000   # 125 nm per count
START_DIRECTION = +1   # +1 starts at the right edge, -1 at the left
HOME_BACKOFF_COUNTS = 4156   # 0 starts exactly on the edge, 8000 counts = 1 mm
HOME_SPEED_COUNTS = 50_000   # counts/s, about 6 mm/s
HOME_ACCEL_COUNTS = 1_000_000   # counts/s², also used for every step

# Close the previous connection if this cell is re-run
if "stage" in globals():
    stage.close()

stage = Stage(STAGE_PORT, COUNTS_PER_MM)
stage.connect()
stage.home(START_DIRECTION, HOME_BACKOFF_COUNTS, HOME_SPEED_COUNTS, HOME_ACCEL_COUNTS)

## 3. Plan the scan

Builds the list of positions and prints the plan. Nothing moves. Check the output before running Section 4, and re-run this cell after any change.

Set **one** of these and leave the other as `None`:

- `N_POSITIONS`: number of positions across `SWEEP_LENGTH_MM`. Every step is equal. Use an odd number so the middle position sits exactly at center (101 positions puts position 50 at `x_center = 0`).
- `STEP_MM`: a fixed step size. If it doesn't divide `SWEEP_LENGTH_MM` evenly, the last step comes up short and the plan names it, along with the settings that would drop it.

Keep the step under λ/4 (about 1.23 mm) to avoid grating lobes. The plan warns if it's larger.

The plan also prints an estimated scan time. Most of it is recording, which is `FRAMES_PER_POSITION` × the frame period (about 130 ms) at each position.

In [ ]:
# Positions (set one, leave the other as None)
N_POSITIONS = 33
STEP_MM = None

# Scan
SWEEP_LENGTH_MM = 100.0
FRAMES_PER_POSITION = 20
SETTLE_S = 0.3   # pause after each step before recording
STEP_SPEED_COUNTS = 50_000   # counts/s, about 6 mm/s

plan = plan_steps(SWEEP_LENGTH_MM, stage.counts_per_mm, FRAMES_PER_POSITION, SETTLE_S,
                  STEP_SPEED_COUNTS, n_positions=N_POSITIONS, step_mm=STEP_MM,
                  frame_period=radar.frame_period)

## 4. Record a scan

Set `SCAN_NAME` and run. Repeat for each scan. Repeated names get `_2`, `_3`... instead of overwriting.

The scan runs on its own. At each position the stage moves, settles, and the radar records `FRAMES_PER_POSITION` frames into that position's own file. Position 0 is the start edge, and every later position is an exact move from there, so the steps don't build up error.

If you interrupt the kernel, the stage stops and every position finished so far stays saved and logged.

**Saved to `DATA_DIR/SESSION_FOLDER/bins/SCAN_NAME/`:**

- `pos_000.bin`, `pos_001.bin`, ... hold the raw radar frames for each position.
- `SCAN_NAME_positions.csv` has one row per position:
  - `position`, `file`: the position index and its `.bin`
  - `target_mm`: planned distance from the start edge
  - `galil_counts`, `galil_counts_after`: encoder position before and after recording (they should match)
  - `x_mm`: actual distance from the start edge
  - `x_center_mm`: actual position from the aperture center, + toward the start edge
  - `n_frames`, `time`: frames saved and when recording started

A summary row is added to `DATA_DIR/SESSION_FOLDER/stepped_log.csv`.

In [ ]:
# Output: /DATA_DIR/SESSION_FOLDER/bin/SCAN_NAME/<.bin files>
DATA_DIR = "data"
SESSION_FOLDER = "stepped_test"
SCAN_NAME = "settle"

scan_dir = record_stepped(radar, stage, plan, DATA_DIR, SESSION_FOLDER, SCAN_NAME)

## 5. End session

Stops the stage and closes both connections. Run this before restarting the kernel or closing VS Code so the radar sees a clean disconnect, which helps avoid the connection hang in 1b next time. Safe to run even if nothing is connected.

In [ ]:
radar.close()
stage.close()

## 6. Export to CSV (optional)

Converts one position's `.bin` into a long-format CSV with one row per (frame, bin, chirp, channel) value and separate `I` and `Q` columns, saved next to the `.bin` with `_long.csv` added. 2D mode only. Doesn't need the radar or stage connected, just the imports cell.

`EXPORT_PATH` defaults to the center position of the last scan from Section 4. For any other file, set it to a path such as `"data/stepped_test/bins/test_scan/pos_000.bin"`.

Each RADC frame is 32,768 rows, so a 20-frame position is about 655,000 rows.

In [ ]:
EXPORT_PATH = f"{scan_dir}/pos_{len(plan['targets']) // 2:03d}.bin"   # center of the last scan, or a full path to any .bin
EXPORT_STREAMS = ["radc"]   # "radc" and/or "rfft", only if they were recorded

export_long_csv(EXPORT_PATH, EXPORT_STREAMS)

## Appendix. Stage calibration (one-time)

Only needed when the stage, encoder, or limit switches change. Run Section 2 first so the stage is connected and homed.

Drives to the far limit switch, counts the travel between both switches, then returns to the start edge. With about 101 mm of travel, roughly 810,000 counts means 125 nm per count (`COUNTS_PER_MM = 8_000`), and roughly 1,021,000 counts means 100 nm per count (`COUNTS_PER_MM = 10_000`).

Last measured on this stage: 808,382 counts (101.05 mm). A 100 mm scan leaves about 1 mm of unused travel at the far end, so the scan never touches the far switch.

In [ ]:
stage.measure_full_travel()